# Generate simulation inputs

This notebook builds the simulation inputs from the Tabula Muris Senis FACS data.

It generates three simulation sets:

1. **Original simulation set**: preserves the existing primary outputs:
   - `simulation_data/adata/TMS_FACS_10k_DE.h5ad`
   - `simulation_data/gs_de_overlap/*.gs`
   - `simulation_data/predictions.csv`
2. **Varying cell-percentage requirement**: fixes the number of causal clusters at 3 and varies the minimum fraction of cells required for each causal cluster over 1%, 2%, 3%, 4%, and 5%.
3. **Varying causal genes per cluster**: fixes the number of causal clusters at 3 and varies causal genes per cluster over 25, 50, 75, and 100.

The original output locations are intentionally unchanged so downstream scripts that already expect those files can continue to use them.


In [1]:
from __future__ import annotations

import hashlib
import math
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc


In [2]:
# === scDRS-FM reproduction: portable path anchor (repo-relative) ===
import os as _os
from pathlib import Path as _Path
def _find_repo_root():
    env = _os.environ.get("SCDRSFM_BASE")
    if env:
        return _Path(env)
    here = _Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "scDRS-FM-main").is_dir() and (d / "scripts").is_dir():
            return d
    return here
_REPO = _find_repo_root()
_DATA = _REPO / "data"
# TMS_FACS 10k subset produced by scripts/00_make_10k_subsets.sh:
_TMS_10K = _DATA / "subsets_10k" / "TMS_FACS" / "TMS_FACS.h5ad"
# Simulation outputs live under data/simulation_data (consumed by nb03 + sim scoring):
_SIM_BASE = _DATA / "simulation_data"

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Keep the original primary output paths unchanged unless downstream code has
# also been updated.

params = dict(
    # input
    input_h5ad=str(_TMS_10K),
    # Alternative previously used input:
    # input_h5ad="tms_data/10k_samples/TMS_FACS.ncell_10k.h5ad",

    # reproducibility / subsampling
    target_n=10_000,
    seed=0,

    # clustering
    leiden_key="leiden",
    leiden_resolution=1.0,

    # differential expression cache
    n_top_de=2000,
    de_methods=["logreg", "t-test", "wilcoxon"],
    de_ref_method="wilcoxon",
    use_raw=False,
    layer=None,

    # original simulation defaults
    min_cells=500,
    geneset_size=1000,
    overlap=50,  # causal genes contributed by each causal donor cluster
    n_replicates=3,
    replicate_start=0,
    max_original_causal_clusters=5,

    # new simulation grids
    fixed_n_causal_clusters=3,
    cell_percent_grid=[1, 2, 3, 4, 5],
    causal_genes_grid=[25, 50, 75, 100],

    # outputs
    out_base=str(_SIM_BASE),
    dataset_prefix="TMS_FACS",
    out_h5ad=str(_SIM_BASE / "adata" / "TMS_FACS_10k_DE.h5ad"),
)

paths = dict(
    base=Path(params["out_base"]),
    original_gs=Path(params["out_base"]) / "gs_de_overlap",
    cell_pct_gs=Path(params["out_base"]) / "gs_cell_pct",
    causal_genes_gs=Path(params["out_base"]) / "gs_causal_genes",
    original_predictions=Path(params["out_base"]) / "predictions.csv",
    cell_pct_predictions=Path(params["out_base"]) / "predictions_cell_pct.csv",
    causal_genes_predictions=Path(params["out_base"]) / "predictions_causal_genes.csv",
    out_h5ad=Path(params["out_h5ad"]),
)


## Shared utility functions

In [3]:
def stable_int_hash(value: str) -> int:
    """Return the same 32-bit integer hash on every Python run."""
    return int.from_bytes(hashlib.md5(value.encode("utf-8")).digest()[:4], "little")


def seed_from_parts(base_seed: int, *parts: object) -> int:
    """Build a deterministic NumPy seed from a base seed and arbitrary labels."""
    payload = "||".join(str(part) for part in parts)
    digest = hashlib.md5(payload.encode("utf-8")).digest()
    hashed = int.from_bytes(digest[:8], "little")
    return int((int(base_seed) + hashed) % (2**32))


def cluster_sort_key(label: object) -> tuple[int, object]:
    """Sort numeric-looking cluster labels numerically, others lexicographically."""
    text = str(label)
    try:
        return (0, int(text))
    except ValueError:
        return (1, text)


def sorted_cluster_labels(labels) -> list[str]:
    return sorted([str(label) for label in labels], key=cluster_sort_key)


def as_join_key(label: object) -> int | str:
    """Keep the historical numeric cluster join key when labels are integers."""
    text = str(label)
    try:
        return int(text)
    except ValueError:
        return text


def write_geneset_file(gs_path: Path, trait: str, geneset: list[str]) -> None:
    """Write a two-column .gs file with the original TRAIT/GENESET layout."""
    gs_path.parent.mkdir(parents=True, exist_ok=True)
    with open(gs_path, "w", encoding="utf-8") as handle:
        handle.write("TRAIT\tGENESET\n")
        handle.write(f"{trait}\t{','.join(geneset)}\n")


def format_trait(
    target: str,
    replicate: int,
    causal_genes_per_cluster: int,
    n_causal_clusters: int,
    donor_clusters: list[str],
    extra_segments: list[str] | None = None,
) -> str:
    """
    Encode simulation metadata in the trait name.

    The original simulation uses exactly:
      {target}__rep{r}__ov{k}__src{N}__donors{d1}+{d2}+...

    New simulation sets may add an extra segment before donors.
    """
    segments = [
        quote(str(target), safe=""),
        f"rep{int(replicate)}",
        f"ov{int(causal_genes_per_cluster)}",
        f"src{int(n_causal_clusters)}",
    ]
    if extra_segments:
        segments.extend(extra_segments)

    donors = "+".join(quote(str(cluster), safe="") for cluster in donor_clusters)
    segments.append(f"donors{donors}")
    return "__".join(segments)


def expm1_matrix_copy(matrix):
    """Return expm1(matrix) without mutating the original matrix."""
    if sp.issparse(matrix):
        copied = matrix.tocsr(copy=True)
        if copied.data.size:
            copied.data = np.expm1(copied.data)
        copied.eliminate_zeros()
        return copied
    return np.expm1(np.asarray(matrix, dtype=float))


## Load, preprocess, subsample, and cluster cells

In [4]:
# Load the full AnnData object.
adata = sc.read_h5ad(params["input_h5ad"])
print(f"Loaded {params['input_h5ad']}: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

# Match the original preprocessing choices.
sc.pp.filter_cells(adata, min_genes=250)
sc.pp.filter_genes(adata, min_cells=50)
sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
sc.pp.log1p(adata)
print(f"After filtering/preprocessing: {adata.n_obs:,} cells x {adata.n_vars:,} genes")


Loaded tms_data/tabula_muris_senis/TMS_FACS.h5ad: 110,824 cells x 22,966 genes
After filtering/preprocessing: 110,824 cells x 20,565 genes


In [5]:
# Subsample to the target cell count.
rng = np.random.default_rng(params["seed"])
if adata.n_obs <= params["target_n"]:
    adata_10k = adata.copy()
    print(f"adata has {adata.n_obs:,} cells (<= {params['target_n']:,}); using the full object.")
else:
    idx = rng.choice(adata.n_obs, size=params["target_n"], replace=False)
    adata_10k = adata[idx].copy()
    print(f"Subsampled adata from {adata.n_obs:,} to {adata_10k.n_obs:,} cells.")

# Compute Leiden labels. If neighbors/PCA are missing, build a minimal graph first.
if "neighbors" not in adata_10k.uns:
    if "X_pca" not in adata_10k.obsm:
        sc.pp.pca(adata_10k, random_state=params["seed"])
    sc.pp.neighbors(adata_10k)

sc.tl.leiden(
    adata_10k,
    resolution=float(params["leiden_resolution"]),
    key_added=params["leiden_key"],
    random_state=params["seed"],
)
print(
    f"Computed Leiden clustering in obs['{params['leiden_key']}'] "
    f"at resolution={params['leiden_resolution']}."
)


Subsampled adata from 110,824 to 10,000 cells.


/tmp/ipykernel_45956/2895299421.py:17: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(


Computed Leiden clustering in obs['leiden'] at resolution=1.0.


## Cache top up-regulated DE genes per Leiden cluster

Each cluster gets a cached list of up-regulated genes for each DE method. The simulation pools use the consensus intersection across `logreg`, `t-test`, and `wilcoxon`, ordered by the reference method (`wilcoxon`).


In [6]:
def cache_top_up_de_genes(adata_obj, params: dict) -> dict[str, dict[str, list[str]]]:
    """Run DE methods and cache top up-regulated genes in adata.uns."""
    label_key = params["leiden_key"]
    n_top_de = int(params["n_top_de"])
    methods = list(params["de_methods"])

    top_de_by_method: dict[str, dict[str, list[str]]] = {}

    for method in methods:
        de_key = f"rank_genes_groups_{label_key}_{method}"
        top_key = f"top_de_genes_up_{label_key}_{n_top_de}_{method}"

        sc.tl.rank_genes_groups(
            adata_obj,
            groupby=label_key,
            method=method,
            n_genes=n_top_de,
            use_raw=params["use_raw"],
            layer=params["layer"],
            key_added=de_key,
        )

        rg = adata_obj.uns[de_key]
        names = rg["names"]
        scores = rg.get("scores", None)

        if not (isinstance(names, np.ndarray) and names.dtype.names is not None):
            raise TypeError(f"Unexpected format for adata.uns['{de_key}']['names'].")
        if scores is None:
            raise KeyError(f"adata.uns['{de_key}'] has no 'scores'; cannot determine up-regulated genes.")

        method_top: dict[str, list[str]] = {}
        for group in names.dtype.names:
            gene_list = np.asarray(names[group], dtype=object)
            score_list = np.asarray(scores[group], dtype=float)
            keep = np.isfinite(score_list) & (score_list > 0)

            up_genes = [str(gene) for gene, keep_gene in zip(gene_list, keep) if keep_gene and gene is not None]
            up_genes = list(dict.fromkeys(up_genes))  # unique, preserving order
            method_top[str(group)] = up_genes[:n_top_de]

        adata_obj.uns[top_key] = method_top
        top_de_by_method[method] = method_top
        print(f"Cached {method:>8} top UP DE genes in adata.uns['{top_key}'].")

    return top_de_by_method


top_de_by_method = cache_top_up_de_genes(adata_10k, params)


Cached   logreg top UP DE genes in adata.uns['top_de_genes_up_leiden_2000_logreg'].


/home/aturcan/miniconda3/envs/scdrs6/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:429: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/home/aturcan/miniconda3/envs/scdrs6/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:431: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/home/aturcan/miniconda3/envs/scdrs6/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:434: P

Cached   t-test top UP DE genes in adata.uns['top_de_genes_up_leiden_2000_t-test'].
Cached wilcoxon top UP DE genes in adata.uns['top_de_genes_up_leiden_2000_wilcoxon'].


/home/aturcan/miniconda3/envs/scdrs6/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:429: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/home/aturcan/miniconda3/envs/scdrs6/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:431: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/home/aturcan/miniconda3/envs/scdrs6/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:434: P

## Save the processed 10k AnnData object

The primary H5AD output path is unchanged. The matrix is saved after `expm1`, matching the original notebook's behavior, but the in-memory `adata_10k` remains log-transformed for the rest of the notebook.


In [7]:
def write_expm1_h5ad(adata_obj, out_path: Path) -> None:
    """Write an AnnData copy with X transformed by expm1."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    adata_to_save = adata_obj.copy()
    adata_to_save.X = expm1_matrix_copy(adata_to_save.X)
    adata_to_save.write_h5ad(out_path, compression="gzip")
    print(f"Wrote expm1-applied AnnData: {out_path}")


write_expm1_h5ad(adata_10k, paths["out_h5ad"])


Wrote expm1-applied AnnData: simulation_data/adata/TMS_FACS_10k_DE.h5ad


## Simulation helper functions

In [8]:
ORIGINAL_PREDICTION_COLUMNS = [
    "dataset_prefix",
    "cluster",
    "replicate",
    "src_n",
    "overlap_k",
    "trait",
    "trait_donor_clusters",
    "trait_donor_clusters_n",
    "trait_donor_clusters_other",
    "trait_donor_clusters_other_n",
    "target_cluster",
    "cluster_key",
    "n_causal_cells",
    "causal_cell_fraction",
    "max_overlap_with",
    "max_overlap_count",
    "max_overlap_pct",
    "max_overlap_percent",
    "n_top_de",
    "n_top_de_window",
    "geneset_size",
    "feasible_src_n",
    "rep_start",
    "n_replicates",
    "n_cells_total",
    "min_cells",
    "gs_path",
    "h5ad_path",
]


def get_cached_top_de(adata_obj, params: dict) -> dict[str, dict[str, list[str]]]:
    """Read cached DE lists from adata.uns."""
    label_key = params["leiden_key"]
    n_top_de = int(params["n_top_de"])
    top_de: dict[str, dict[str, list[str]]] = {}

    for method in params["de_methods"]:
        key = f"top_de_genes_up_{label_key}_{n_top_de}_{method}"
        if key not in adata_obj.uns:
            raise KeyError(f"Missing adata.uns['{key}']; run the DE caching cell first.")
        top_de[method] = {str(cluster): list(genes) for cluster, genes in adata_obj.uns[key].items()}

    return top_de


def get_count_map(adata_obj, label_key: str) -> dict[str, int]:
    labels = adata_obj.obs[label_key].astype(str).to_numpy()
    uniq, counts = np.unique(labels, return_counts=True)
    return {str(label): int(count) for label, count in zip(uniq, counts)}


def get_eligible_clusters(
    adata_obj,
    label_key: str,
    top_de_by_method: dict[str, dict[str, list[str]]],
    min_cells: int,
    include_equal: bool = False,
) -> tuple[list[str], dict[str, int]]:
    """Return clusters that satisfy the cell-count rule and have DE caches for all methods."""
    count_map = get_count_map(adata_obj, label_key)
    clusters_common = set(next(iter(top_de_by_method.values())).keys())
    for method_top in top_de_by_method.values():
        clusters_common &= set(method_top.keys())

    labels = sorted_cluster_labels(count_map.keys())
    if include_equal:
        eligible = [label for label in labels if count_map[label] >= min_cells and label in clusters_common]
    else:
        # Strictly greater than min_cells preserves the original notebook behavior.
        eligible = [label for label in labels if count_map[label] > min_cells and label in clusters_common]
    return eligible, count_map


def build_consensus_top_pools(
    top_de_by_method: dict[str, dict[str, list[str]]],
    eligible: list[str],
    n_top_de: int,
    ref_method: str,
) -> tuple[dict[str, list[str]], int]:
    """Build per-cluster consensus DE pools: logreg ∩ t-test ∩ wilcoxon."""
    methods = list(top_de_by_method.keys())
    if ref_method not in methods:
        raise ValueError(f"Reference method {ref_method!r} is not present in cached DE methods: {methods}")

    min_len = min(len(top_de_by_method[method][cluster]) for method in methods for cluster in eligible)
    n_top_de_window = min(int(n_top_de), min_len) if int(n_top_de) > 0 else min_len
    if n_top_de_window <= 0:
        raise ValueError("n_top_de_window is <= 0; check cached DE gene lists.")

    sliced = {
        method: {cluster: top_de_by_method[method][cluster][:n_top_de_window] for cluster in eligible}
        for method in methods
    }

    consensus: dict[str, list[str]] = {}
    for cluster in eligible:
        intersection = set(sliced[methods[0]][cluster])
        for method in methods[1:]:
            intersection &= set(sliced[method][cluster])
        consensus[cluster] = [gene for gene in sliced[ref_method][cluster] if gene in intersection]

    return consensus, n_top_de_window


def get_background_candidates(adata_obj, top_de_by_method: dict[str, dict[str, list[str]]], n_top_de: int) -> list[str]:
    """Genes not in any cluster's top DE list for any method."""
    top_union = set()
    for method_top in top_de_by_method.values():
        for genes in method_top.values():
            top_union.update(genes[: int(n_top_de)])
    return [gene for gene in adata_obj.var_names.to_list() if gene not in top_union]


def prepare_simulation_context(
    adata_obj,
    params: dict,
    min_cells: int,
    include_equal: bool = False,
) -> dict:
    """Collect reusable objects needed to generate a simulation set."""
    label_key = params["leiden_key"]
    top_de = get_cached_top_de(adata_obj, params)
    eligible, count_map = get_eligible_clusters(
        adata_obj,
        label_key=label_key,
        top_de_by_method=top_de,
        min_cells=int(min_cells),
        include_equal=include_equal,
    )
    if not eligible:
        comparator = ">=" if include_equal else ">"
        raise ValueError(f"No eligible clusters with {comparator} {min_cells} cells.")

    consensus_top, n_top_de_window = build_consensus_top_pools(
        top_de_by_method=top_de,
        eligible=eligible,
        n_top_de=int(params["n_top_de"]),
        ref_method=params["de_ref_method"],
    )
    background_candidates = get_background_candidates(adata_obj, top_de, params["n_top_de"])

    return dict(
        adata=adata_obj,
        label_key=label_key,
        top_de_by_method=top_de,
        eligible=eligible,
        count_map=count_map,
        consensus_top=consensus_top,
        n_top_de_window=n_top_de_window,
        background_candidates=background_candidates,
        min_cells=int(min_cells),
        include_equal=bool(include_equal),
        n_cells_total=int(adata_obj.n_obs),
    )


def summarize_context(title: str, ctx: dict) -> None:
    comparator = ">=" if ctx["include_equal"] else ">"
    total_unique = set()
    print(f"\n{title}")
    print(f"Eligible clusters: {len(ctx['eligible'])} with {comparator} {ctx['min_cells']} cells")
    print(f"Consensus window: top {ctx['n_top_de_window']} genes per method")
    for cluster in ctx["eligible"]:
        n_candidates = len(ctx["consensus_top"][cluster])
        total_unique.update(ctx["consensus_top"][cluster])
        print(f"  cluster {cluster}: {ctx['count_map'][cluster]:,} cells, {n_candidates:,} consensus genes")
    print(f"Total unique consensus genes across eligible clusters: {len(total_unique):,}")
    print(f"Background candidates: {len(ctx['background_candidates']):,}")


def validate_causal_gene_pools(ctx: dict, causal_genes_per_cluster: int) -> None:
    """Ensure each eligible cluster has enough consensus genes for one donor contribution."""
    too_small = {
        cluster: len(ctx["consensus_top"][cluster])
        for cluster in ctx["eligible"]
        if len(ctx["consensus_top"][cluster]) < int(causal_genes_per_cluster)
    }
    if too_small:
        details = ", ".join(f"{cluster}({n})" for cluster, n in sorted(too_small.items(), key=lambda x: cluster_sort_key(x[0])))
        raise ValueError(
            f"Not enough consensus genes to sample {causal_genes_per_cluster} causal genes per cluster: {details}. "
            "Increase n_top_de or decrease causal_genes_per_cluster."
        )


def feasible_src_n_values(
    n_eligible: int,
    geneset_size: int,
    causal_genes_per_cluster: int,
    n_background_candidates: int,
    max_src_n: int,
) -> list[int]:
    """Return feasible numbers of causal clusters under current sampling constraints."""
    values = []
    upper = min(int(max_src_n), int(n_eligible))
    for src_n in range(1, upper + 1):
        n_causal_genes = int(src_n) * int(causal_genes_per_cluster)
        n_background = int(geneset_size) - n_causal_genes
        if n_background < 0:
            continue
        if n_background > int(n_background_candidates):
            continue
        values.append(src_n)
    return values


def choose_donor_clusters(rrng: np.random.Generator, eligible: list[str], target: str, n_causal_clusters: int) -> list[str]:
    """Choose the target plus n_causal_clusters - 1 additional eligible donor clusters."""
    n_causal_clusters = int(n_causal_clusters)
    if n_causal_clusters < 1:
        raise ValueError("n_causal_clusters must be >= 1")
    if n_causal_clusters == 1:
        return [str(target)]

    other_clusters = [cluster for cluster in eligible if str(cluster) != str(target)]
    if len(other_clusters) < n_causal_clusters - 1:
        raise ValueError(
            f"Need {n_causal_clusters - 1} non-target clusters, but only found {len(other_clusters)}."
        )
    picked_others = rrng.choice(other_clusters, size=n_causal_clusters - 1, replace=False).tolist()
    return [str(target)] + [str(cluster) for cluster in picked_others]


def sample_unique_from_pool(
    rrng: np.random.Generator,
    pool: list[str],
    k: int,
    used: set[str],
) -> list[str]:
    """Sample k genes from pool after excluding genes already used in this geneset."""
    available = [gene for gene in pool if gene not in used]
    if len(available) < int(k):
        raise ValueError(
            f"Not enough available genes to sample k={k} after excluding used genes "
            f"(have {len(available)} available)."
        )
    picked = rrng.choice(available, size=int(k), replace=False).tolist()
    used.update(picked)
    return picked


def sample_geneset_from_donors(
    rrng: np.random.Generator,
    donor_clusters: list[str],
    consensus_top: dict[str, list[str]],
    background_candidates: list[str],
    causal_genes_per_cluster: int,
    geneset_size: int,
) -> list[str]:
    """Sample causal genes from donor clusters and fill the rest from background genes."""
    causal_genes_per_cluster = int(causal_genes_per_cluster)
    geneset_size = int(geneset_size)
    n_causal_total = causal_genes_per_cluster * len(donor_clusters)
    n_background = geneset_size - n_causal_total

    if n_background < 0:
        raise ValueError(
            f"geneset_size={geneset_size} is smaller than required causal genes "
            f"({n_causal_total})."
        )
    if n_background > len(background_candidates):
        raise ValueError(
            f"Need {n_background} background genes, but only {len(background_candidates)} are available."
        )

    used: set[str] = set()
    causal_genes: list[str] = []
    for donor in donor_clusters:
        causal_genes.extend(
            sample_unique_from_pool(
                rrng=rrng,
                pool=consensus_top[str(donor)],
                k=causal_genes_per_cluster,
                used=used,
            )
        )

    background_genes = rrng.choice(background_candidates, size=n_background, replace=False).tolist()
    geneset = causal_genes + background_genes
    rrng.shuffle(geneset)
    return geneset


def donor_count_strings(donor_clusters: list[str], count_map: dict[str, int], n_cells_total: int) -> tuple[str, str]:
    counts = [int(count_map[str(cluster)]) for cluster in donor_clusters]
    fractions = [count / float(n_cells_total) if n_cells_total else np.nan for count in counts]
    return ";".join(str(count) for count in counts), ";".join(f"{fraction:.6g}" for fraction in fractions)


def make_record(
    *,
    ctx: dict,
    target: str,
    replicate: int,
    n_causal_clusters: int,
    causal_genes_per_cluster: int,
    donor_clusters: list[str],
    trait: str,
    gs_path: Path,
    feasible_src_n: list[int],
    extra: dict | None = None,
) -> dict:
    return dict(
        context=ctx,
        target=str(target),
        replicate=int(replicate),
        src_n=int(n_causal_clusters),
        overlap_k=int(causal_genes_per_cluster),
        donor_clusters=[str(cluster) for cluster in donor_clusters],
        trait=str(trait),
        gs_path=str(gs_path),
        feasible_src_n=list(feasible_src_n),
        extra=extra or {},
    )


def build_prediction_row(record: dict, params: dict) -> dict:
    ctx = record["context"]
    target = str(record["target"])
    donors = [str(cluster) for cluster in record["donor_clusters"]]
    count_map = ctx["count_map"]
    consensus_top = ctx["consensus_top"]
    eligible = ctx["eligible"]
    n_top_de_window = int(ctx["n_top_de_window"])

    n_causal = int(count_map.get(target, 0))
    causal_frac = n_causal / float(ctx["n_cells_total"]) if ctx["n_cells_total"] else np.nan

    target_set = set(consensus_top[target])
    overlaps = {
        cluster: len(target_set.intersection(consensus_top[cluster]))
        for cluster in eligible
        if str(cluster) != target
    }
    if overlaps:
        max_overlap_with = max(overlaps, key=overlaps.get)
        max_overlap_count = int(overlaps[max_overlap_with])
        max_overlap_pct = max_overlap_count / float(n_top_de_window) if n_top_de_window else 0.0
    else:
        max_overlap_with = None
        max_overlap_count = 0
        max_overlap_pct = 0.0

    donors_other = [cluster for cluster in donors if cluster != target]

    row = dict(
        dataset_prefix=params["dataset_prefix"],
        cluster=as_join_key(target),
        replicate=int(record["replicate"]),
        src_n=int(record["src_n"]),
        overlap_k=int(record["overlap_k"]),
        trait=record["trait"],
        trait_donor_clusters=";".join(donors),
        trait_donor_clusters_n=len(donors),
        trait_donor_clusters_other=";".join(donors_other),
        trait_donor_clusters_other_n=len(donors_other),
        target_cluster=target,
        cluster_key=params["leiden_key"],
        n_causal_cells=n_causal,
        causal_cell_fraction=causal_frac,
        max_overlap_with=max_overlap_with,
        max_overlap_count=max_overlap_count,
        max_overlap_pct=max_overlap_pct,
        max_overlap_percent=100.0 * max_overlap_pct,
        n_top_de=int(params["n_top_de"]),
        n_top_de_window=n_top_de_window,
        geneset_size=int(params["geneset_size"]),
        feasible_src_n=";".join(str(x) for x in record["feasible_src_n"]),
        rep_start=int(params["replicate_start"]),
        n_replicates=int(params["n_replicates"]),
        n_cells_total=int(ctx["n_cells_total"]),
        min_cells=int(ctx["min_cells"]),
        gs_path=record["gs_path"],
        h5ad_path=str(paths["out_h5ad"]),
    )
    row.update(record.get("extra", {}))
    return row


def records_to_prediction_table(
    records: list[dict],
    params: dict,
    extra_columns: list[str] | None = None,
    sort_by_extra: tuple[str, ...] = (),
) -> pd.DataFrame:
    rows = [build_prediction_row(record, params) for record in records]

    def row_sort_key(row: dict):
        extra_values = tuple(row.get(column, -1) for column in sort_by_extra)
        return extra_values + cluster_sort_key(row["target_cluster"]) + (
            int(row["src_n"]),
            int(row["replicate"]),
            int(row["overlap_k"]),
        )

    rows = sorted(rows, key=row_sort_key)
    columns = ORIGINAL_PREDICTION_COLUMNS.copy()
    if extra_columns is None:
        extra_columns = sorted({key for row in rows for key in row.keys()} - set(columns))
    columns.extend([column for column in extra_columns if column not in columns])
    return pd.DataFrame(rows, columns=columns)


def write_prediction_table(pred: pd.DataFrame, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    pred.to_csv(out_path, index=False)
    print(f"Wrote predictions: {out_path} ({len(pred):,} rows)")


## 1. Original simulation set

This preserves the primary original output locations and the original legacy seed formula. It varies the number of causal clusters from 1 to 5, subject to feasibility under the current eligible clusters and background gene pool.


In [9]:
def generate_original_simulations(adata_obj, params: dict) -> pd.DataFrame:
    ctx = prepare_simulation_context(
        adata_obj,
        params,
        min_cells=int(params["min_cells"]),
        include_equal=False,  # preserves original strict > min_cells eligibility
    )
    if len(ctx["eligible"]) < 2:
        raise ValueError(
            f"Need at least 2 eligible Leiden clusters with > {params['min_cells']} cells; "
            f"found {len(ctx['eligible'])}."
        )

    causal_genes_per_cluster = int(params["overlap"])
    validate_causal_gene_pools(ctx, causal_genes_per_cluster)

    feasible_src_n = feasible_src_n_values(
        n_eligible=len(ctx["eligible"]),
        geneset_size=int(params["geneset_size"]),
        causal_genes_per_cluster=causal_genes_per_cluster,
        n_background_candidates=len(ctx["background_candidates"]),
        max_src_n=int(params["max_original_causal_clusters"]),
    )
    if not feasible_src_n:
        raise ValueError("No feasible original src_n values under the current parameters.")

    summarize_context("Original simulation context", ctx)
    print(f"Feasible original src_n values: {feasible_src_n}")

    records: list[dict] = []
    out_dir = paths["original_gs"]
    rep_start = int(params["replicate_start"])
    rep_stop = rep_start + int(params["n_replicates"])

    for src_n in feasible_src_n:
        n_background = int(params["geneset_size"]) - causal_genes_per_cluster * src_n
        print(f"\nGenerating original genesets for src_n={src_n}; background genes per set={n_background}")

        for target in ctx["eligible"]:
            for replicate in range(rep_start, rep_stop):
                # Preserve the original seed formula for primary outputs.
                rrng = np.random.default_rng(
                    int(params["seed"])
                    + 10_000
                    + stable_int_hash(str(target))
                    + int(replicate)
                    + 1_000_000 * int(src_n)
                )
                donor_clusters = choose_donor_clusters(rrng, ctx["eligible"], target, src_n)
                geneset = sample_geneset_from_donors(
                    rrng=rrng,
                    donor_clusters=donor_clusters,
                    consensus_top=ctx["consensus_top"],
                    background_candidates=ctx["background_candidates"],
                    causal_genes_per_cluster=causal_genes_per_cluster,
                    geneset_size=int(params["geneset_size"]),
                )

                trait = format_trait(
                    target=target,
                    replicate=replicate,
                    causal_genes_per_cluster=causal_genes_per_cluster,
                    n_causal_clusters=src_n,
                    donor_clusters=donor_clusters,
                )
                gs_path = out_dir / f"{params['dataset_prefix']}_{target}_{replicate}_{causal_genes_per_cluster}_src{src_n}.gs"
                write_geneset_file(gs_path, trait, geneset)

                records.append(
                    make_record(
                        ctx=ctx,
                        target=target,
                        replicate=replicate,
                        n_causal_clusters=src_n,
                        causal_genes_per_cluster=causal_genes_per_cluster,
                        donor_clusters=donor_clusters,
                        trait=trait,
                        gs_path=gs_path,
                        feasible_src_n=feasible_src_n,
                    )
                )

    pred = records_to_prediction_table(records, params, extra_columns=[])
    write_prediction_table(pred, paths["original_predictions"])
    return pred


original_predictions = generate_original_simulations(adata_10k, params)
original_predictions.head()



Original simulation context
Eligible clusters: 5 with > 500 cells
Consensus window: top 2000 genes per method
  cluster 0: 902 cells, 1,047 consensus genes
  cluster 1: 868 cells, 867 consensus genes
  cluster 2: 682 cells, 956 consensus genes
  cluster 3: 584 cells, 1,020 consensus genes
  cluster 4: 572 cells, 901 consensus genes
Total unique consensus genes across eligible clusters: 3,809
Background candidates: 1,448
Feasible original src_n values: [1, 2, 3, 4, 5]

Generating original genesets for src_n=1; background genes per set=950

Generating original genesets for src_n=2; background genes per set=900

Generating original genesets for src_n=3; background genes per set=850

Generating original genesets for src_n=4; background genes per set=800

Generating original genesets for src_n=5; background genes per set=750
Wrote predictions: simulation_data/predictions.csv (75 rows)


,dataset_prefix,cluster,replicate,src_n,overlap_k,trait,trait_donor_clusters,trait_donor_clusters_n,trait_donor_clusters_other,trait_donor_clusters_other_n,...,n_top_de,n_top_de_window,geneset_size,feasible_src_n,rep_start,n_replicates,n_cells_total,min_cells,gs_path,h5ad_path
0,TMS_FACS,0,0,1,50,0__rep0__ov50__src1__donors0,0,1,,0,...,2000,2000,1000,1;2;3;4;5,0,3,10000,500,simulation_data/gs_de_overlap/TMS_FACS_0_0_50_...,simulation_data/adata/TMS_FACS_10k_DE.h5ad
1,TMS_FACS,0,1,1,50,0__rep1__ov50__src1__donors0,0,1,,0,...,2000,2000,1000,1;2;3;4;5,0,3,10000,500,simulation_data/gs_de_overlap/TMS_FACS_0_1_50_...,simulation_data/adata/TMS_FACS_10k_DE.h5ad
2,TMS_FACS,0,2,1,50,0__rep2__ov50__src1__donors0,0,1,,0,...,2000,2000,1000,1;2;3;4;5,0,3,10000,500,simulation_data/gs_de_overlap/TMS_FACS_0_2_50_...,simulation_data/adata/TMS_FACS_10k_DE.h5ad
3,TMS_FACS,0,0,2,50,0__rep0__ov50__src2__donors0+2,0;2,2,2,1,...,2000,2000,1000,1;2;3;4;5,0,3,10000,500,simulation_data/gs_de_overlap/TMS_FACS_0_0_50_...,simulation_data/adata/TMS_FACS_10k_DE.h5ad
4,TMS_FACS,0,1,2,50,0__rep1__ov50__src2__donors0+3,0;3,2,3,1,...,2000,2000,1000,1;2;3;4;5,0,3,10000,500,simulation_data/gs_de_overlap/TMS_FACS_0_1_50_...,simulation_data/adata/TMS_FACS_10k_DE.h5ad


## 2. Simulation set varying the cell-percentage requirement

This set fixes the number of causal clusters at 3. For each requested percentage, every causal donor cluster must have at least that percentage of cells in `adata_10k`.

Outputs:
- genesets: `simulation_data/gs_cell_pct/*.gs`
- predictions: `simulation_data/predictions_cell_pct.csv`


In [10]:
CELL_PCT_EXTRA_COLUMNS = [
    "simulation_set",
    "n_causal_clusters",
    "causal_genes_per_cluster",
    "min_cell_percent",
    "min_cells_required",
    "donor_cell_counts",
    "donor_cell_fractions",
]


def generate_cell_percent_simulations(adata_obj, params: dict) -> pd.DataFrame:
    n_causal_clusters = int(params["fixed_n_causal_clusters"])
    causal_genes_per_cluster = int(params["overlap"])
    rep_start = int(params["replicate_start"])
    rep_stop = rep_start + int(params["n_replicates"])
    records: list[dict] = []

    for pct in params["cell_percent_grid"]:
        pct = int(pct)
        min_cells_required = int(math.ceil((pct / 100.0) * int(adata_obj.n_obs)))
        ctx = prepare_simulation_context(
            adata_obj,
            params,
            min_cells=min_cells_required,
            include_equal=True,  # "at least pct%" for the new grid
        )

        if len(ctx["eligible"]) < n_causal_clusters:
            print(
                f"Skipping cell_pct={pct}: need {n_causal_clusters} eligible clusters, "
                f"found {len(ctx['eligible'])}."
            )
            continue

        validate_causal_gene_pools(ctx, causal_genes_per_cluster)
        n_background = int(params["geneset_size"]) - causal_genes_per_cluster * n_causal_clusters
        if n_background < 0 or n_background > len(ctx["background_candidates"]):
            print(
                f"Skipping cell_pct={pct}: background requirement {n_background} is infeasible "
                f"with {len(ctx['background_candidates'])} background genes."
            )
            continue

        summarize_context(f"Cell-percent simulation context: {pct}%", ctx)
        feasible_src_n = [n_causal_clusters]

        for target in ctx["eligible"]:
            for replicate in range(rep_start, rep_stop):
                rrng = np.random.default_rng(
                    seed_from_parts(params["seed"], "cell_pct", pct, target, replicate, n_causal_clusters)
                )
                donor_clusters = choose_donor_clusters(rrng, ctx["eligible"], target, n_causal_clusters)
                geneset = sample_geneset_from_donors(
                    rrng=rrng,
                    donor_clusters=donor_clusters,
                    consensus_top=ctx["consensus_top"],
                    background_candidates=ctx["background_candidates"],
                    causal_genes_per_cluster=causal_genes_per_cluster,
                    geneset_size=int(params["geneset_size"]),
                )

                trait = format_trait(
                    target=target,
                    replicate=replicate,
                    causal_genes_per_cluster=causal_genes_per_cluster,
                    n_causal_clusters=n_causal_clusters,
                    donor_clusters=donor_clusters,
                    extra_segments=[f"cellpct{pct}"],
                )
                gs_path = (
                    paths["cell_pct_gs"]
                    / f"{params['dataset_prefix']}_{target}_{replicate}_pct{pct}_ov{causal_genes_per_cluster}_src{n_causal_clusters}.gs"
                )
                write_geneset_file(gs_path, trait, geneset)

                donor_counts, donor_fractions = donor_count_strings(
                    donor_clusters,
                    ctx["count_map"],
                    ctx["n_cells_total"],
                )
                records.append(
                    make_record(
                        ctx=ctx,
                        target=target,
                        replicate=replicate,
                        n_causal_clusters=n_causal_clusters,
                        causal_genes_per_cluster=causal_genes_per_cluster,
                        donor_clusters=donor_clusters,
                        trait=trait,
                        gs_path=gs_path,
                        feasible_src_n=feasible_src_n,
                        extra=dict(
                            simulation_set="vary_cell_percent",
                            n_causal_clusters=n_causal_clusters,
                            causal_genes_per_cluster=causal_genes_per_cluster,
                            min_cell_percent=pct,
                            min_cells_required=min_cells_required,
                            donor_cell_counts=donor_counts,
                            donor_cell_fractions=donor_fractions,
                        ),
                    )
                )

    if not records:
        raise ValueError("No cell-percent simulation records were generated.")

    pred = records_to_prediction_table(
        records,
        params,
        extra_columns=CELL_PCT_EXTRA_COLUMNS,
        sort_by_extra=("min_cell_percent",),
    )
    write_prediction_table(pred, paths["cell_pct_predictions"])
    return pred


cell_pct_predictions = generate_cell_percent_simulations(adata_10k, params)
cell_pct_predictions.head()



Cell-percent simulation context: 1%
Eligible clusters: 31 with >= 100 cells
Consensus window: top 1717 genes per method
  cluster 0: 902 cells, 904 consensus genes
  cluster 1: 868 cells, 776 consensus genes
  cluster 2: 682 cells, 847 consensus genes
  cluster 3: 584 cells, 894 consensus genes
  cluster 4: 572 cells, 822 consensus genes
  cluster 5: 455 cells, 736 consensus genes
  cluster 6: 381 cells, 1,063 consensus genes
  cluster 7: 371 cells, 923 consensus genes
  cluster 8: 357 cells, 1,067 consensus genes
  cluster 9: 343 cells, 926 consensus genes
  cluster 10: 290 cells, 898 consensus genes
  cluster 11: 276 cells, 171 consensus genes
  cluster 12: 273 cells, 497 consensus genes
  cluster 13: 230 cells, 954 consensus genes
  cluster 14: 229 cells, 1,050 consensus genes
  cluster 15: 229 cells, 837 consensus genes
  cluster 16: 210 cells, 964 consensus genes
  cluster 17: 205 cells, 1,128 consensus genes
  cluster 18: 204 cells, 629 consensus genes
  cluster 19: 197 cells, 1

,dataset_prefix,cluster,replicate,src_n,overlap_k,trait,trait_donor_clusters,trait_donor_clusters_n,trait_donor_clusters_other,trait_donor_clusters_other_n,...,min_cells,gs_path,h5ad_path,simulation_set,n_causal_clusters,causal_genes_per_cluster,min_cell_percent,min_cells_required,donor_cell_counts,donor_cell_fractions
0,TMS_FACS,0,0,3,50,0__rep0__ov50__src3__cellpct1__donors0+16+2,0;16;2,3,16;2,2,...,100,simulation_data/gs_cell_pct/TMS_FACS_0_0_pct1_...,simulation_data/adata/TMS_FACS_10k_DE.h5ad,vary_cell_percent,3,50,1,100,902;210;682,0.0902;0.021;0.0682
1,TMS_FACS,0,1,3,50,0__rep1__ov50__src3__cellpct1__donors0+13+24,0;13;24,3,13;24,2,...,100,simulation_data/gs_cell_pct/TMS_FACS_0_1_pct1_...,simulation_data/adata/TMS_FACS_10k_DE.h5ad,vary_cell_percent,3,50,1,100,902;230;167,0.0902;0.023;0.0167
2,TMS_FACS,0,2,3,50,0__rep2__ov50__src3__cellpct1__donors0+23+10,0;23;10,3,23;10,2,...,100,simulation_data/gs_cell_pct/TMS_FACS_0_2_pct1_...,simulation_data/adata/TMS_FACS_10k_DE.h5ad,vary_cell_percent,3,50,1,100,902;173;290,0.0902;0.0173;0.029
3,TMS_FACS,1,0,3,50,1__rep0__ov50__src3__cellpct1__donors1+21+17,1;21;17,3,21;17,2,...,100,simulation_data/gs_cell_pct/TMS_FACS_1_0_pct1_...,simulation_data/adata/TMS_FACS_10k_DE.h5ad,vary_cell_percent,3,50,1,100,868;179;205,0.0868;0.0179;0.0205
4,TMS_FACS,1,1,3,50,1__rep1__ov50__src3__cellpct1__donors1+10+11,1;10;11,3,10;11,2,...,100,simulation_data/gs_cell_pct/TMS_FACS_1_1_pct1_...,simulation_data/adata/TMS_FACS_10k_DE.h5ad,vary_cell_percent,3,50,1,100,868;290;276,0.0868;0.029;0.0276


## 3. Simulation set varying causal genes per cluster

This set fixes the number of causal clusters at 3 and uses the original `min_cells` criterion. It varies the number of causal genes contributed by each causal donor cluster over 25, 50, 75, and 100.

For each target/replicate, the donor clusters are held constant across causal-gene counts so the gene-count grid changes the causal gene count rather than the donor-cluster composition.

Outputs:
- genesets: `simulation_data/gs_causal_genes/*.gs`
- predictions: `simulation_data/predictions_causal_genes.csv`


In [11]:
CAUSAL_GENES_EXTRA_COLUMNS = [
    "simulation_set",
    "n_causal_clusters",
    "causal_genes_per_cluster",
    "min_cells_required",
    "donor_cell_counts",
    "donor_cell_fractions",
]


def generate_causal_gene_count_simulations(adata_obj, params: dict) -> pd.DataFrame:
    n_causal_clusters = int(params["fixed_n_causal_clusters"])
    rep_start = int(params["replicate_start"])
    rep_stop = rep_start + int(params["n_replicates"])

    ctx = prepare_simulation_context(
        adata_obj,
        params,
        min_cells=int(params["min_cells"]),
        include_equal=False,  # same strict rule as original simulations
    )
    if len(ctx["eligible"]) < n_causal_clusters:
        raise ValueError(
            f"Need at least {n_causal_clusters} eligible clusters for causal-gene grid; "
            f"found {len(ctx['eligible'])}."
        )
    summarize_context("Causal-gene-count simulation context", ctx)

    # Fix donor clusters across causal-gene counts for each target/replicate.
    donor_cache: dict[tuple[str, int], list[str]] = {}
    for target in ctx["eligible"]:
        for replicate in range(rep_start, rep_stop):
            donor_rng = np.random.default_rng(
                seed_from_parts(params["seed"], "causal_genes_donors", target, replicate, n_causal_clusters)
            )
            donor_cache[(target, replicate)] = choose_donor_clusters(
                donor_rng,
                ctx["eligible"],
                target,
                n_causal_clusters,
            )

    records: list[dict] = []
    feasible_src_n = [n_causal_clusters]

    for causal_genes_per_cluster in params["causal_genes_grid"]:
        causal_genes_per_cluster = int(causal_genes_per_cluster)
        validate_causal_gene_pools(ctx, causal_genes_per_cluster)

        n_background = int(params["geneset_size"]) - causal_genes_per_cluster * n_causal_clusters
        if n_background < 0 or n_background > len(ctx["background_candidates"]):
            print(
                f"Skipping causal_genes_per_cluster={causal_genes_per_cluster}: "
                f"background requirement {n_background} is infeasible with "
                f"{len(ctx['background_candidates'])} background genes."
            )
            continue

        print(
            f"\nGenerating causal-gene-count genesets for {causal_genes_per_cluster} causal genes/cluster; "
            f"background genes per set={n_background}"
        )

        for target in ctx["eligible"]:
            for replicate in range(rep_start, rep_stop):
                donor_clusters = donor_cache[(target, replicate)]
                gene_rng = np.random.default_rng(
                    seed_from_parts(
                        params["seed"],
                        "causal_genes_sampling",
                        causal_genes_per_cluster,
                        target,
                        replicate,
                        n_causal_clusters,
                    )
                )
                geneset = sample_geneset_from_donors(
                    rrng=gene_rng,
                    donor_clusters=donor_clusters,
                    consensus_top=ctx["consensus_top"],
                    background_candidates=ctx["background_candidates"],
                    causal_genes_per_cluster=causal_genes_per_cluster,
                    geneset_size=int(params["geneset_size"]),
                )

                trait = format_trait(
                    target=target,
                    replicate=replicate,
                    causal_genes_per_cluster=causal_genes_per_cluster,
                    n_causal_clusters=n_causal_clusters,
                    donor_clusters=donor_clusters,
                )
                gs_path = (
                    paths["causal_genes_gs"]
                    / f"{params['dataset_prefix']}_{target}_{replicate}_ov{causal_genes_per_cluster}_src{n_causal_clusters}.gs"
                )
                write_geneset_file(gs_path, trait, geneset)

                donor_counts, donor_fractions = donor_count_strings(
                    donor_clusters,
                    ctx["count_map"],
                    ctx["n_cells_total"],
                )
                records.append(
                    make_record(
                        ctx=ctx,
                        target=target,
                        replicate=replicate,
                        n_causal_clusters=n_causal_clusters,
                        causal_genes_per_cluster=causal_genes_per_cluster,
                        donor_clusters=donor_clusters,
                        trait=trait,
                        gs_path=gs_path,
                        feasible_src_n=feasible_src_n,
                        extra=dict(
                            simulation_set="vary_causal_genes",
                            n_causal_clusters=n_causal_clusters,
                            causal_genes_per_cluster=causal_genes_per_cluster,
                            min_cells_required=int(params["min_cells"]),
                            donor_cell_counts=donor_counts,
                            donor_cell_fractions=donor_fractions,
                        ),
                    )
                )

    if not records:
        raise ValueError("No causal-gene-count simulation records were generated.")

    pred = records_to_prediction_table(
        records,
        params,
        extra_columns=CAUSAL_GENES_EXTRA_COLUMNS,
        sort_by_extra=("causal_genes_per_cluster",),
    )
    write_prediction_table(pred, paths["causal_genes_predictions"])
    return pred


causal_genes_predictions = generate_causal_gene_count_simulations(adata_10k, params)
causal_genes_predictions.head()



Causal-gene-count simulation context
Eligible clusters: 5 with > 500 cells
Consensus window: top 2000 genes per method
  cluster 0: 902 cells, 1,047 consensus genes
  cluster 1: 868 cells, 867 consensus genes
  cluster 2: 682 cells, 956 consensus genes
  cluster 3: 584 cells, 1,020 consensus genes
  cluster 4: 572 cells, 901 consensus genes
Total unique consensus genes across eligible clusters: 3,809
Background candidates: 1,448

Generating causal-gene-count genesets for 25 causal genes/cluster; background genes per set=925

Generating causal-gene-count genesets for 50 causal genes/cluster; background genes per set=850

Generating causal-gene-count genesets for 75 causal genes/cluster; background genes per set=775

Generating causal-gene-count genesets for 100 causal genes/cluster; background genes per set=700
Wrote predictions: simulation_data/predictions_causal_genes.csv (60 rows)


,dataset_prefix,cluster,replicate,src_n,overlap_k,trait,trait_donor_clusters,trait_donor_clusters_n,trait_donor_clusters_other,trait_donor_clusters_other_n,...,n_cells_total,min_cells,gs_path,h5ad_path,simulation_set,n_causal_clusters,causal_genes_per_cluster,min_cells_required,donor_cell_counts,donor_cell_fractions
0,TMS_FACS,0,0,3,25,0__rep0__ov25__src3__donors0+4+2,0;4;2,3,4;2,2,...,10000,500,simulation_data/gs_causal_genes/TMS_FACS_0_0_o...,simulation_data/adata/TMS_FACS_10k_DE.h5ad,vary_causal_genes,3,25,500,902;572;682,0.0902;0.0572;0.0682
1,TMS_FACS,0,1,3,25,0__rep1__ov25__src3__donors0+3+1,0;3;1,3,3;1,2,...,10000,500,simulation_data/gs_causal_genes/TMS_FACS_0_1_o...,simulation_data/adata/TMS_FACS_10k_DE.h5ad,vary_causal_genes,3,25,500,902;584;868,0.0902;0.0584;0.0868
2,TMS_FACS,0,2,3,25,0__rep2__ov25__src3__donors0+4+3,0;4;3,3,4;3,2,...,10000,500,simulation_data/gs_causal_genes/TMS_FACS_0_2_o...,simulation_data/adata/TMS_FACS_10k_DE.h5ad,vary_causal_genes,3,25,500,902;572;584,0.0902;0.0572;0.0584
3,TMS_FACS,1,0,3,25,1__rep0__ov25__src3__donors1+4+0,1;4;0,3,4;0,2,...,10000,500,simulation_data/gs_causal_genes/TMS_FACS_1_0_o...,simulation_data/adata/TMS_FACS_10k_DE.h5ad,vary_causal_genes,3,25,500,868;572;902,0.0868;0.0572;0.0902
4,TMS_FACS,1,1,3,25,1__rep1__ov25__src3__donors1+2+3,1;2;3,3,2;3,2,...,10000,500,simulation_data/gs_causal_genes/TMS_FACS_1_1_o...,simulation_data/adata/TMS_FACS_10k_DE.h5ad,vary_causal_genes,3,25,500,868;682;584,0.0868;0.0682;0.0584


## Output summary

In [12]:
summary = pd.DataFrame(
    [
        dict(
            simulation_set="original",
            geneset_dir=str(paths["original_gs"]),
            predictions_csv=str(paths["original_predictions"]),
            n_prediction_rows=len(original_predictions),
        ),
        dict(
            simulation_set="vary_cell_percent",
            geneset_dir=str(paths["cell_pct_gs"]),
            predictions_csv=str(paths["cell_pct_predictions"]),
            n_prediction_rows=len(cell_pct_predictions),
        ),
        dict(
            simulation_set="vary_causal_genes",
            geneset_dir=str(paths["causal_genes_gs"]),
            predictions_csv=str(paths["causal_genes_predictions"]),
            n_prediction_rows=len(causal_genes_predictions),
        ),
    ]
)
summary


,simulation_set,geneset_dir,predictions_csv,n_prediction_rows
0,original,simulation_data/gs_de_overlap,simulation_data/predictions.csv,75
1,vary_cell_percent,simulation_data/gs_cell_pct,simulation_data/predictions_cell_pct.csv,213
2,vary_causal_genes,simulation_data/gs_causal_genes,simulation_data/predictions_causal_genes.csv,60
